In [26]:
import pandas as pd


In [27]:
df = pd.read_csv('/content/Follow-up_Records.csv')

In [28]:
print(df.head())

   patient_id  visit_date  age_years  weight_kg   bmi  systolic_bp_mmHg  \
0  P-2025-001  2024-02-15         52       83.7  28.3               138   
1  P-2025-001  2024-03-15         52       83.4  28.2               147   
2  P-2025-001  2024-04-15         52       83.1  28.1               140   
3  P-2025-001  2024-05-15         52       83.0  28.1               136   
4  P-2025-001  2024-06-15         52       82.6  27.9               133   

   diastolic_bp_mmHg  heart_rate_bpm  body_temp_C  fasting_glucose_mg_dL  ...  \
0                 86              80         36.8                    137  ...   
1                 89              80         37.0                    140  ...   
2                 84              76         36.8                    122  ...   
3                 88              77         36.8                    112  ...   
4                 88              78         36.8                    101  ...   

   diet_quality_score_0_100  sleep_hours  exercise_sessions_pe

In [29]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import numpy as np

In [30]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(sparse_output=False)
cat_encoded = encoder.fit_transform(df[cat_cols])

scaler = MinMaxScaler(feature_range=(-1, 1))
num_scaled = scaler.fit_transform(df[num_cols])

# combine processed data
data_processed = np.hstack((num_scaled, cat_encoded))

In [31]:
import torch
import torch.nn as nn

data_dim = data_processed.shape[1]  # total features
latent_dim = 64  # size of random noise input

# generator
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, data_dim),
            nn.Tanh()  # output in range [-1, 1]
        )
    def forward(self, z):
        return self.model(z)

# discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(data_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()  # probability of real/fake
        )
    def forward(self, x):
        return self.model(x)

In [32]:
from torch.utils.data import DataLoader, TensorDataset

# convert data to PyTorch tensors
real_data = torch.tensor(data_processed, dtype=torch.float32)
dataset = TensorDataset(real_data)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# initialize models
generator = Generator()
discriminator = Discriminator()

# optimizers
lr = 0.0002
optim_G = torch.optim.Adam(generator.parameters(), lr=lr)
optim_D = torch.optim.Adam(discriminator.parameters(), lr=lr)

# loss
criterion = nn.BCELoss()

epochs = 2000
for epoch in range(epochs):
    for real_batch, in loader:
        batch_size = real_batch.size(0)

        # labels for real and fake data
        real_labels = torch.ones((batch_size, 1))
        fake_labels = torch.zeros((batch_size, 1))

        # train discriminator
        z = torch.randn(batch_size, latent_dim)
        fake_data = generator(z)

        real_loss = criterion(discriminator(real_batch), real_labels)
        fake_loss = criterion(discriminator(fake_data.detach()), fake_labels)
        d_loss = (real_loss + fake_loss) / 2

        optim_D.zero_grad()
        d_loss.backward()
        optim_D.step()

        # train generator
        z = torch.randn(batch_size, latent_dim)
        fake_data = generator(z)
        g_loss = criterion(discriminator(fake_data), real_labels)  # want fake to be real

        optim_G.zero_grad()
        g_loss.backward()
        optim_G.step()

    if epoch % 200 == 0:
        print(f"Epoch [{epoch}/{epochs}]  D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

Epoch [0/2000]  D_loss: 0.6969  G_loss: 0.6906
Epoch [200/2000]  D_loss: 0.1458  G_loss: 2.3554
Epoch [400/2000]  D_loss: 0.1479  G_loss: 1.7312
Epoch [600/2000]  D_loss: 0.0996  G_loss: 2.5823
Epoch [800/2000]  D_loss: 0.1618  G_loss: 1.2211
Epoch [1000/2000]  D_loss: 0.4003  G_loss: 1.7994
Epoch [1200/2000]  D_loss: 0.0462  G_loss: 1.6606
Epoch [1400/2000]  D_loss: 0.1137  G_loss: 2.1167
Epoch [1600/2000]  D_loss: 0.0337  G_loss: 3.1754
Epoch [1800/2000]  D_loss: 0.0397  G_loss: 3.4101


In [33]:
# generate new synthetic data
z = torch.randn(10, latent_dim)  # 10 synthetic samples
synthetic_data_scaled = generator(z).detach().numpy()

# inverse transform
num_synthetic = scaler.inverse_transform(synthetic_data_scaled[:, :len(num_cols)])
cat_synthetic = encoder.inverse_transform(synthetic_data_scaled[:, len(num_cols):])

# combine into dataframe
synthetic_df = pd.DataFrame(num_synthetic, columns=num_cols)
synthetic_df[cat_cols] = cat_synthetic

print(synthetic_df)

   age_years  weight_kg        bmi  systolic_bp_mmHg  diastolic_bp_mmHg  \
0  53.000000  80.715080  27.305235        120.015137          78.802872   
1  52.999996  80.771973  27.316750        120.167542          76.221756   
2  52.999775  80.898041  27.357637        121.002075          77.461044   
3  52.999992  80.752663  27.314980        120.070122          79.135468   
4  52.616673  82.036652  27.594269        131.961395          78.923935   
5  53.000000  80.714592  27.305136        120.016327          77.377655   
6  52.011574  81.888672  27.644167        125.524124          87.926849   
7  52.641003  81.730652  27.600576        127.881416          82.225792   
8  52.871346  81.631424  27.605253        126.610199          79.965271   
9  52.862247  81.528847  27.531242        126.086693          81.158623   

   heart_rate_bpm  body_temp_C  fasting_glucose_mg_dL  \
0       78.864609    36.709976              92.708206   
1       76.245369    36.739647              94.729759   
2  